# Pipeline 高级配置指南

本教程介绍 ModelPipeline 的高级功能：

1. **PipelineConfigs**: 自定义每个模型的初始化、训练和预测参数
2. **模型筛选**: include_models / exclude_models
3. **自定义 Scaler**: 使用不同的数据缩放器
4. **自定义评估指标**: 替换默认的 MAE
5. **model_init_kwargs**: 通过双下划线语法传递模型参数
6. **模型查询与配置获取**

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 200
dates = pd.date_range(start='2020-01-01', periods=n, freq='D')
values = np.sin(np.linspace(0, 4 * np.pi, n)) + np.random.randn(n) * 0.1
data = pd.DataFrame({'date': dates, 'value': values})

LAGS = 12

## 1. PipelineConfigs 自定义配置

PipelineConfigs 允许你为同一个模型创建多个变体，每个变体使用不同的超参数。

In [ ]:
from PipelineTS.pipeline import ModelPipeline, PipelineConfigs

# 创建配置: 两个 LightGBM 变体
configs = PipelineConfigs([
    ('lightgbm', 'lgbm_small', {
        'init_configs': {'n_estimators': 50},
        'fit_configs': {}
    }),
    ('lightgbm', 'lgbm_large', {
        'init_configs': {'n_estimators': 300},
        'fit_configs': {}
    }),
])

In [ ]:
pipeline = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    include_models=['lightgbm'],
    configs=configs,
    quantile=None, cv=3
)

leaderboard = pipeline.fit(data)
leaderboard

## 2. 模型筛选

通过 `include_models` 和 `exclude_models` 控制训练哪些模型。

In [ ]:
# 查看所有可用模型
print("所有可用模型:")
print(ModelPipeline.list_all_available_models())

# 预定义模型集合:
# 'light' - 轻量级模型（默认，速度优先）
# 'all'   - 所有模型
# 'nn'    - 仅神经网络模型
# 'ml'    - 仅机器学习模型

In [ ]:
# 仅使用 ML 模型
pipeline_ml = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    include_models='ml', quantile=None, cv=2
)
leaderboard_ml = pipeline_ml.fit(data)
print("ML 模型排行榜:")
leaderboard_ml

In [ ]:
# 指定具体模型列表
pipeline_custom = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    include_models=['lightgbm', 'xgboost', 'd_linear', 'n_linear'],
    quantile=None, cv=2
)
leaderboard_custom = pipeline_custom.fit(data)
leaderboard_custom

## 3. 自定义 Scaler

默认使用 MinMaxScaler，你可以替换为任何 sklearn TransformerMixin 兼容的缩放器。

In [ ]:
from sklearn.preprocessing import StandardScaler
from PipelineTS.preprocessing import Scaler

# 方法 1: 使用 sklearn 的 StandardScaler
pipeline_std = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    include_models=['lightgbm'],
    scaler=StandardScaler(),
    quantile=None, cv=2
)

# 方法 2: 使用 PipelineTS 内置的 Scaler
# 支持: 'min_max', 'standard', 'quantile', 'gauss_rank'
scaler = Scaler('gauss_rank')

# 方法 3: 关闭数据缩放
pipeline_no_scale = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    include_models=['lightgbm'],
    scaler=None,   # 不使用缩放器
    quantile=None, cv=2
)

## 4. 自定义评估指标

In [ ]:
from PipelineTS.spinesTS.metrics import rmse, wmape

# 使用 RMSE 作为评估指标
pipeline_rmse = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    include_models=['lightgbm', 'xgboost'],
    metric=rmse,                # 自定义指标函数
    metric_less_is_better=True, # RMSE 越小越好
    quantile=None, cv=2
)
leaderboard_rmse = pipeline_rmse.fit(data)
leaderboard_rmse

## 5. 通过双下划线语法传递模型参数

使用 `模型名__参数名=值` 的格式传递模型初始化参数。

In [ ]:
pipeline_kwargs = ModelPipeline(
    time_col='date', target_col='value', lags=LAGS,
    include_models=['lightgbm', 'xgboost'],
    quantile=None, cv=2,
    lightgbm__n_estimators=100,    # LightGBM 专属参数
    xgboost__n_estimators=150,     # XGBoost 专属参数
    xgboost__verbose=0
)
leaderboard_kwargs = pipeline_kwargs.fit(data)
leaderboard_kwargs

## 6. 模型查询与配置获取

In [ ]:
# 获取最佳模型
best_model = pipeline_kwargs.get_model()
print(f"最佳模型: {type(best_model).__name__}")

# 获取指定模型
model_name = pipeline_kwargs.leader_board_.iloc[0]['model']
specific_model = pipeline_kwargs.get_model(model_name)
print(f"指定模型: {model_name}")

# 获取模型全部配置
configs = pipeline_kwargs.get_model_all_configs()
print(f"模型配置: {configs}")

In [ ]:
# 使用指定模型进行预测
result = pipeline_kwargs.predict(10, model_name=model_name)
result